In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import logging
import os
from datetime import datetime


# =========================================================
# Widgets
# =========================================================
params = {
    "source_path": "",
    "target_path": "",
    "archive_path": "",
    "logs_path": ""
}

for k, v in params.items():
    dbutils.widgets.text(k, v)

sourcepath = dbutils.widgets.get("source_path")
targetpath = dbutils.widgets.get("target_path")
archivepath = dbutils.widgets.get("archive_path")
logspath = dbutils.widgets.get("logs_path")

# =========================================================
# Validations
# =========================================================
if not sourcepath:
    raise Exception("source_path is empty")

if not targetpath:
    raise Exception("target_path is empty")

if not archivepath:
    raise Exception("archive_path is empty")

if not logspath:
    raise Exception("logs_path is empty")

# =========================================================
# Logging Setup
# =========================================================
logspath_local = logspath

os.makedirs(logspath_local, exist_ok=True)

logs_file = os.path.join(
    logspath_local,
    f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_employeesload.log"
)

logger = logging.getLogger()

for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

logging.basicConfig(
    filename=logs_file,
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("Job Started")

logging.debug(f"sourcepath: {sourcepath}")
logging.debug(f"targetpath: {targetpath}")
logging.debug(f"archivepath: {archivepath}")
logging.debug(f"logspath: {logspath}")

# =========================================================
# Archive File
# =========================================================

def archive_file(file_path,file_name):
    try:

        archive_file_path = (
            f"{archivepath.rstrip('/')}/{file_name}"
        )

        dbutils.fs.mv(file_path, archive_file_path)

        logging.info(
            f"{file_name} archived successfully to "
            f"{archive_file_path}"
        )

    except Exception as archive_error:

        logging.error(
            f"Error archiving file {file_name}: "
            f"{str(archive_error)}"
        )


try:

    # =========================================================
    # Read Files
    # =========================================================
    files = dbutils.fs.ls(sourcepath)

    files = [f for f in files if f.path.endswith(".csv")]

    sorted_files = sorted(
        files,
        key=lambda x: x.modificationTime,
        reverse=False
    )

    logging.info(f"Total files to be loaded: {len(sorted_files)}")
    # =========================================================
    # Check Files Available
    # =========================================================
    if sorted_files:

        logging.info(f"CSV files found in path: {sourcepath}")

        for f in sorted_files:

            logging.info(f"Processing file: {f.name}")

            delta_exists = False

            try:
                
                delta_table = DeltaTable.forPath(spark,targetpath)

                delta_table.toDF().count()
                
                delta_exists = True

                logging.info(
                    "Delta table already exists"
                )

            except Exception as e:

                delta_exists = False

                logging.info(
                    "Delta table does not exist"
                )

                logging.debug(
                    f"Delta validation exception: "
                    f"{str(e)}"
                )


            # =========================================================
            # Initial Load
            # =========================================================
            if not delta_exists:

                logging.info("Initial Load Started")

                df = spark.read.format("csv") \
                    .options(header=True, inferSchema=True) \
                    .load(f.path)

                row_count = df.count()

                logging.info(f"Rows loaded from source: {row_count}")

                initial_df = df \
                    .withColumn("last_modified", F.current_timestamp()) \
                    .withColumn("isactive", F.lit(1))

                initial_df.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .save(targetpath)

                logging.info("Initial Load Completed")

                # =========================================================
                # Archive File
                # =========================================================
                archive_file(f.path, f.name)

            # =========================================================
            # Incremental Load
            # =========================================================
            else:

                logging.info("Incremental Load Started")

                target_df = spark.read.format("delta").load(targetpath)

                logging.info(
                    f"Total rows in target table: "
                    f"{target_df.count()}"
                )

                cols = set(target_df.columns)

                # =========================================================
                # Add SCD Columns if Missing
                # =========================================================
                if "last_modified" not in cols:

                    logging.info(
                        "Adding column: last_modified"
                    )

                    spark.sql(f"""
                    ALTER TABLE delta.`{targetpath}`
                    ADD COLUMNS (last_modified TIMESTAMP)
                    """)

                    spark.sql(f"""
                    UPDATE delta.`{targetpath}`
                    SET last_modified = current_timestamp()
                    WHERE last_modified IS NULL
                    """)

                if "isactive" not in cols:

                    logging.info(
                        "Adding column: isactive"
                    )

                    spark.sql(f"""
                    ALTER TABLE delta.`{targetpath}`
                    ADD COLUMNS (isactive INT)
                    """)

                    spark.sql(f"""
                    UPDATE delta.`{targetpath}`
                    SET isactive = 1
                    WHERE isactive IS NULL
                    """)

                # =========================================================
                # Reload Target Table
                # =========================================================
                target_df = spark.read.format("delta").load(targetpath)

                active_target_df = target_df.filter(
                    "isactive = 1"
                )

                logging.info(
                    f"Active rows in target table: "
                    f"{active_target_df.count()}"
                )

                # =========================================================
                # Read Source File
                # =========================================================
                schema = target_df.schema

                source_df = spark.read.format("csv") \
                    .schema(schema) \
                    .option("header", True) \
                    .load(f.path)

                logging.info(
                    f"Rows in source file: "
                    f"{source_df.count()}"
                )

                # =========================================================
                # Add SCD Columns
                # =========================================================
                source_df = source_df \
                    .withColumn(
                        "last_modified",
                        F.current_timestamp()
                    ) \
                    .withColumn(
                        "isactive",
                        F.lit(1)
                    )

                # =========================================================
                # Join Source & Target
                # =========================================================
                joined = source_df.alias("s").join(
                    active_target_df.alias("t"),
                    on="emp_id",
                    how="left"
                )

                # =========================================================
                # New Records
                # =========================================================
                new_records = joined.filter(
                    F.col("t.emp_id").isNull()
                ).select("s.*")

                logging.info(
                    f"New records count: "
                    f"{new_records.count()}"
                )

                # =========================================================
                # Changed Records
                # =========================================================
                compare_cols = [
                    c for c in source_df.columns
                    if c not in (
                        "last_modified",
                        "isactive"
                    )
                ]

                change_condition = " OR ".join(
                    [
                        f"s.{c} != t.{c}"
                        for c in compare_cols
                    ]
                )

                updated_records = joined.filter(
                    F.col("t.emp_id").isNotNull() &
                    F.expr(change_condition)
                ).select("s.*")

                logging.info(
                    f"Updated records count: "
                    f"{updated_records.count()}"
                )

                # =========================================================
                # Temp View
                # =========================================================
                updated_records.createOrReplaceTempView(
                    "updated_records"
                )

                # =========================================================
                # Inactivate Old Records
                # =========================================================
                spark.sql(f"""
                UPDATE delta.`{targetpath}`
                SET isactive = 0,
                    last_modified = current_timestamp()
                WHERE emp_id IN (
                    SELECT emp_id
                    FROM updated_records
                )
                AND isactive = 1
                """)

                logging.info(
                    "Existing records marked inactive"
                )

                # =========================================================
                # Merge New + Updated Records
                # =========================================================
                final_df = new_records.unionByName(
                    updated_records
                )

                logging.info(
                    f"Total rows to insert: "
                    f"{final_df.count()}"
                )

                # =========================================================
                # Write Data
                # =========================================================
                final_df.write \
                    .format("delta") \
                    .mode("append") \
                    .option("mergeSchema", True) \
                    .save(targetpath)

                logging.info(
                    "Incremental Load Completed"
                )

                # =========================================================
                # Archive File
                # =========================================================
                archive_file(f.path,f.name)

            logging.info(
                    f"Completed processing file: "
                    f"{f.name}"
                )
        logging.info(
            f"All files are processed: "
            f"{len(sorted_files)}"
        )
        dbutils.fs.rm(sourcepath,
        True
        )
        logging.info("All files except CSV are removed")

        os.makedirs(sourcepath, exist_ok=True)

        logging.info(f"sourcepath is created {sourcepath}")
    else:

        logging.info(
            f"No CSV files found in path: {sourcepath}"
        )

except Exception as e:

    logging.exception(
        f"Job Failed: {str(e)}"
    )

    raise

finally:

    logging.info("Job Ended")

    logging.shutdown()

    print(f"Log File Created: {logs_file}")